In [ ]:
from google.colab import drive
# 挂载Google Drive，以便访问存储在其中的数据和文件
drive.mount('/content/drive')

# 设置您的数据存放的基础目录路径。
# 请确保将您的股票数据文件（分钟线、资金流、日线等）放在这个目录下。
base_dir = '/content/drive/MyDrive/Colab Notebooks/ant_data'

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 📁 路径设置（Colab环境下建议挂载 Google Drive）
base_dir = '/content/drive/MyDrive/Colab Notebooks/ant_data/to_fns'
save_path = "/content/data/"
chart_dir = "/content/charts/"

os.makedirs(save_path, exist_ok=True)
os.makedirs(chart_dir, exist_ok=True)

# ✅ 中文字体支持（Colab 环境）
!apt-get update -qq > /dev/null
!apt-get install fonts-wqy-zenhei -qq > /dev/null

font_path = '/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei']
    plt.rcParams['axes.unicode_minus'] = False
else:
    print(f"⚠️ 未找到中文字体文件，图表可能无法正确显示中文")

# 🔍 主力净流入分析函数（半小时粒度）
def analyze_inflow_extremes_half_hour(df):
    df['datetime'] = pd.to_datetime(df['日期'].astype(str) + ' ' + df['时间'].astype(str))
    df['date'] = df['datetime'].dt.date
    df['half_hour'] = df['datetime'].dt.floor('30min').dt.strftime('%H:%M')

    high_idx = df.groupby('date')['主力净流入'].idxmax()
    low_idx = df.groupby('date')['主力净流入'].idxmin()

    highs = df.loc[high_idx][['date', '主力净流入', 'datetime', 'half_hour']].rename(columns={
        '主力净流入': 'max_inflow', 'datetime': 'max_time', 'half_hour': 'max_half_hour'})
    lows = df.loc[low_idx][['date', '主力净流入', 'datetime', 'half_hour']].rename(columns={
        '主力净流入': 'min_inflow', 'datetime': 'min_time', 'half_hour': 'min_half_hour'})

    result = pd.merge(highs, lows, on='date')
    result['max_before_min'] = result['max_time'] < result['min_time']
    result['inflow_reversal_strength'] = (result['max_inflow'] - result['min_inflow']).abs()

    return result

# 📥 遍历所有 _合并.csv 文件
for filename in os.listdir(base_dir):
    if filename.endswith('_合并.csv'):
        file_path = os.path.join(base_dir, filename)
        code = filename.split('_')[0]
        name = filename.split('_')[1]

        print(f"✅ 正在处理：{code} {name}")
        try:
            df = pd.read_csv(file_path)
            df['datetime'] = pd.to_datetime(df['日期'].astype(str) + ' ' + df['时间'].astype(str))

            # 半小时分析
            result_half_hour = analyze_inflow_extremes_half_hour(df)
            result_half_hour.to_csv(os.path.join(save_path, f"{code}_{name}_主力净流入分析_半小时.csv"), index=False)

            # 可视化（半小时）
            plt.figure(figsize=(12, 6))
            max_counts_half_hour = result_half_hour['max_half_hour'].value_counts().sort_index()
            min_counts_half_hour = result_half_hour['min_half_hour'].value_counts().sort_index()

            ax_half_hour = max_counts_half_hour.plot(kind='bar', alpha=0.6, label='最大流入时间（主力买入）', color='orange')
            min_counts_half_hour.plot(kind='bar', alpha=0.6, label='最小流入时间（主力卖出）', color='blue', ax=ax_half_hour)

            plt.xlabel('时间 (半小时)')
            plt.ylabel('频次')
            plt.title(f'{code} {name} 主力净流入极值时间分布（半小时）')

            half_hour_labels = sorted(set(result_half_hour['max_half_hour']).union(result_half_hour['min_half_hour']))
            ax_half_hour.set_xticks(range(len(half_hour_labels)))
            ax_half_hour.set_xticklabels(half_hour_labels, rotation=90)

            plt.legend()
            plt.grid(axis='y')
            plt.tight_layout()
            plt.savefig(os.path.join(chart_dir, f"{code}_{name}_主力净流入时间分布图_半小时.png"))
            plt.close()

            # 小时分析
            df['hour'] = df['datetime'].dt.floor('h').dt.strftime('%H:00')
            high_idx_hour = df.groupby('date')['主力净流入'].idxmax()
            low_idx_hour = df.groupby('date')['主力净流入'].idxmin()

            highs_hour = df.loc[high_idx_hour][['date', '主力净流入', 'datetime', 'hour']].rename(columns={
                '主力净流入': 'max_inflow', 'datetime': 'max_time', 'hour': 'max_hour'})
            lows_hour = df.loc[low_idx_hour][['date', '主力净流入', 'datetime', 'hour']].rename(columns={
                '主力净流入': 'min_inflow', 'datetime': 'min_time', 'hour': 'min_hour'})

            result_hour = pd.merge(highs_hour, lows_hour, on='date')
            result_hour['max_before_min'] = result_hour['max_time'] < result_hour['min_time']
            result_hour['inflow_reversal_strength'] = (result_hour['max_inflow'] - result_hour['min_inflow']).abs()

            result_hour.to_csv(os.path.join(save_path, f"{code}_{name}_主力净流入分析_小时.csv"), index=False)

            # 可视化（小时）
            plt.figure(figsize=(12, 6))
            max_counts_hour = result_hour['max_hour'].value_counts().sort_index()
            min_counts_hour = result_hour['min_hour'].value_counts().sort_index()

            ax_hour = max_counts_hour.plot(kind='bar', alpha=0.6, label='最大流入时间', color='orange')
            min_counts_hour.plot(kind='bar', alpha=0.6, label='最小流入时间', color='blue', ax=ax_hour)

            plt.xlabel('时间 (小时)')
            plt.ylabel('频次')
            plt.title(f'{code} {name} 主力净流入极值时间分布（小时）')
            plt.xticks(rotation=90)
            plt.legend()
            plt.grid(axis='y')
            plt.tight_layout()
            plt.savefig(os.path.join(chart_dir, f"{code}_{name}_主力净流入时间分布图_小时.png"))
            plt.close()

        except Exception as e:
            print(f"❌ 处理失败：{filename}，错误：{e}")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ 正在处理：000066 中国长城
✅ 正在处理：002008 大族激光
✅ 正在处理：002068 黑猫股份
✅ 正在处理：002276 万马股份
✅ 正在处理：002371 北方华创
✅ 正在处理：002463 沪电股份
✅ 正在处理：002475 立讯精密
✅ 正在处理：002536 飞龙股份
✅ 正在处理：002837 英维克
✅ 正在处理：002850 科达利
✅ 正在处理：002938 鹏鼎控股
✅ 正在处理：002298 中电鑫龙
✅ 正在处理：002364 中恒电气
✅ 正在处理：002269 美邦服饰
✅ 正在处理：002456 欧菲光
✅ 正在处理：601138 工业富联
✅ 正在处理：603920 世运电路
✅ 正在处理：601288 农业银行


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# 📁 路径设置（Colab环境下建议挂载 Google Drive）
base_dir = '/content/drive/MyDrive/Colab Notebooks/ant_data/to_fns'
save_path = "/content/data/"
chart_dir = "/content/charts/"

os.makedirs(save_path, exist_ok=True)
os.makedirs(chart_dir, exist_ok=True)

# 🔍 主力净流入极值分析函数
def analyze_inflow_extremes(df):
    # 合并日期和时间为 datetime
    df['datetime'] = pd.to_datetime(df['日期'].astype(str) + ' ' + df['时间'].astype(str))
    df['date'] = df['datetime'].dt.date

    # 找出每天最大值和最小值的索引
    high_idx = df.groupby('date')['主力净流入'].idxmax()
    low_idx = df.groupby('date')['主力净流入'].idxmin()

    # 提取对应行
    highs = df.loc[high_idx][['date', '主力净流入', 'datetime']].rename(columns={'主力净流入': 'max_inflow', 'datetime': 'max_time'})
    lows = df.loc[low_idx][['date', '主力净流入', 'datetime']].rename(columns={'主力净流入': 'min_inflow', 'datetime': 'min_time'})

    # 合并结果
    result = pd.merge(highs, lows, on='date')

    # 提取小时和分钟字段用于可视化
    result['max_hour'] = result['max_time'].dt.hour + result['max_time'].dt.minute / 60
    result['min_hour'] = result['min_time'].dt.hour + result['min_time'].dt.minute / 60

    # 策略信号：是否先流入再流出
    result['max_before_min'] = result['max_time'] < result['min_time']

    # 可选：资金反转强度评分（绝对值差）
    result['inflow_reversal_strength'] = (result['max_inflow'] - result['min_inflow']).abs()

    return result

# 📥 遍历所有 _合并.csv 文件
for filename in os.listdir(base_dir):
    if filename.endswith('_合并.csv'):
        file_path = os.path.join(base_dir, filename)
        code = filename.split('_')[0]
        name = filename.split('_')[1]

        print(f"✅ 正在处理：{code} {name}")
        df = pd.read_csv(file_path)

        # 检查必要字段
        if '主力净流入' not in df.columns or '日期' not in df.columns or '时间' not in df.columns:
            print(f"⚠️ 文件缺少必要字段，跳过：{filename}")
            continue

        # 分析主力净流入的极值时间
        result = analyze_inflow_extremes(df)

        # 💾 保存结果
        result.to_csv(os.path.join(save_path, f"{code}_{name}_主力净流入分析.csv"), index=False)
        print(result.head())

        # 📊 可视化极值时间分布（颜色与标签一致）
        plt.figure(figsize=(10, 5))

        # 最大净流入时间 → 橙色 → 主力买入行为
        plt.hist(result['max_hour'], bins=24, alpha=0.6, label='最大净流入时间（主力买入）', color='orange')

        # 最小净流入时间 → 蓝色 → 主力卖出行为
        plt.hist(result['min_hour'], bins=24, alpha=0.6, label='最小净流入时间（主力卖出）', color='blue')

        plt.xlabel('时间（小时）')
        plt.ylabel('频次')
        plt.title(f'{code} {name} 主力净流入极值时间分布')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(chart_dir, f"{code}_{name}_主力净流入时间分布图.png"))
        plt.close()


✅ 正在处理：000066 中国长城
         date   max_inflow            max_time  min_inflow  \
0  2023-09-25  26989293040 2023-09-25 14:56:00 -1803237376   
1  2023-09-26  26989293040 2023-09-26 14:56:00 -1803237376   
2  2023-09-27  26989293040 2023-09-27 14:56:00 -1803237376   
3  2023-09-28  26989293040 2023-09-28 14:56:00 -1803237376   
4  2023-10-09  26989293040 2023-10-09 14:56:00 -1803237376   

             min_time   max_hour  min_hour  max_before_min  \
0 2023-09-25 09:34:00  14.933333  9.566667           False   
1 2023-09-26 09:34:00  14.933333  9.566667           False   
2 2023-09-27 09:34:00  14.933333  9.566667           False   
3 2023-09-28 09:34:00  14.933333  9.566667           False   
4 2023-10-09 09:34:00  14.933333  9.566667           False   

   inflow_reversal_strength  
0               28792530416  
1               28792530416  
2               28792530416  
3               28792530416  
4               28792530416  
✅ 正在处理：002008 大族激光
         date   max_inflow          